# Video → 3D Reconstruction (VGGT-Omega + SAM 3)

This notebook walks through the full pipeline:
1. Install the repo and its Python deps (Torch + CUDA come pre-installed on Colab).
2. Authenticate with Hugging Face so we can pull the gated VGGT-Omega and SAM 3 checkpoints.
3. Provide a video, either from Google Drive or by uploading directly.
4. Run the reconstruction CLI: frame extraction → VGGT-Omega geometry → SAM 3 semantics → fused point cloud + viewer.
5. Preview the interactive HTML viewer inline.
6. Run an open-vocabulary text query against the reconstructed scene.
7. Zip the outputs and download them.

## 1. Clone the repo and install Python dependencies
We use `requirements-colab.txt` because Colab already provides matched Torch / CUDA wheels — reinstalling Torch wastes ~3 minutes and sometimes breaks the runtime.

In [ ]:
!git clone https://github.com/ayushmaankaria/Video-to-3D-Reconstruction.git
%cd Video-to-3D-Reconstruction

!pip install -q -r requirements-colab.txt

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Authenticate with Hugging Face
Both VGGT-Omega and SAM 3 are gated. Before running this cell:

1. Request access at https://huggingface.co/facebook/vggt-omega and https://huggingface.co/facebook/sam3 .
2. Create a *Read* token at https://huggingface.co/settings/tokens .
3. In Colab's left sidebar, open the 🔑 **Secrets** panel, add a secret named `HF_TOKEN`, paste your token, and enable notebook access.

The cell below reads that secret and logs in non-interactively.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
assert token, "Add an HF_TOKEN secret in the Colab Secrets panel and re-run."
login(token=token)
print("Logged in to Hugging Face.")

## 3a. (Option A) Use a video from Google Drive
Mount Drive and point `VIDEO_PATH` at your phone video. Skip this cell and use 3b instead if you'd rather upload directly.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

VIDEO_PATH = "/content/drive/MyDrive/desk_video.mp4"  # <-- edit this
!test -f "$VIDEO_PATH" && echo "Using $VIDEO_PATH" || echo "Video not found. Update VIDEO_PATH or use the upload cell below."

## 3b. (Option B) Upload a video directly
Pick a small video (under ~100 MB is fine). The selected file's path becomes `VIDEO_PATH`.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_name = next(iter(uploaded.keys()))
VIDEO_PATH = f"/content/Video-to-3D-Reconstruction/{video_name}"
print("Using", VIDEO_PATH)

## 4. Download the VGGT-Omega checkpoint
We grab the 512-resolution 1B checkpoint into `checkpoints/`. If you'd rather use the 256 + text-alignment variant, swap the repo id and pass `--image-resolution 256 --enable-alignment` in the next cell.

In [ ]:
from huggingface_hub import snapshot_download

ckpt_dir = snapshot_download(
    repo_id="facebook/VGGT-Omega-1B-512",
    local_dir="checkpoints/VGGT-Omega-1B-512",
    local_dir_use_symlinks=False,
)
print("Checkpoint at:", ckpt_dir)

## 5. Run the reconstruction pipeline
This single CLI call does:
1. Sample frames from the video (`--max-frames`, hybrid sharp+uniform mode).
2. Run VGGT-Omega for dense depth + camera intrinsics/extrinsics.
3. Run SAM 3 once per frame for each text concept in `--concepts` and stamp out a label map.
4. Fuse everything into a colored, semantically-labeled point cloud and write `runs/desk/exports/`.

Tweak `--concepts` to match what's actually in your scene — extra concepts cost roughly +0.3 s/frame each.

In [ ]:
!python -m spatial_recon.cli run \
  --video "$VIDEO_PATH" \
  --out runs/desk \
  --checkpoint checkpoints/VGGT-Omega-1B-512/model.pt \
  --image-resolution 512 \
  --max-frames 24 \
  --concepts "desk,chair,monitor,keyboard,mouse,cup,wall,floor" \
  --conf-percentile 35 \
  --sample-stride 2 \
  --voxel-size 0.015

## 6. Preview the interactive viewer inline
`viewer.html` is a self-contained Plotly scene — colored points, semantic toggle, and the camera trajectory. Rendering inside Colab works for clouds up to ~500k points; for bigger ones, download and open locally.

In [ ]:
from IPython.display import HTML, display
display(HTML(filename="runs/desk/exports/viewer.html"))

## 7. Open-vocabulary 3D query (SAM 3 backed)
Re-runs SAM 3 with your text prompt over the original frames, then highlights the top-K% of fused points whose source pixels fall inside the highest-scoring SAM 3 masks. Output is `runs/desk/exports/query_<text>.ply`.

In [ ]:
!python -m spatial_recon.cli query \
  --run runs/desk \
  --text "chair" \
  --topk-percent 8

## 8. Zip and download all outputs
Bundles frames, the VGGT predictions NPZ, SAM 3 label maps, fused PLY/GLB, the viewer HTML, the legend, the query PLY, and `REPORT.md`.

In [ ]:
from google.colab import files
!zip -r desk_reconstruction_outputs.zip runs/desk
files.download("desk_reconstruction_outputs.zip")